# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)

d

# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
from pyspark.sql.functions import sum, col, coalesce, lit

# Get Tables

bookings_df = spark.table("bookings")
members_df = spark.table("members")
facilities_df = spark.table("facilities")

# Transform
result = (
    facilities_df
    .join(
        bookings_df.groupBy("facid").agg(sum("slots").alias("total_slots")),
        on="facid",
        how="left"
    )
    .withColumn("total_slots", coalesce(col("total_slots"), lit(0)))
)

# Load

result.write.mode("overwrite").parquet("dbfs:/FileStore/tables/facHoursByMonth.parquet")


## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# Extract
bookings_df = spark.table("bookings")
members_df = spark.table("members")
facilities_df = spark.table("facilities")

# Transform
from pyspark.sql.functions import col, concat_ws


result = (members_df.join(bookings_df, on="memid", how="inner").join(facilities_df, on="facid", how="inner")
    .filter(col("name").contains("Tennis Court")).select(concat_ws(" ", col("surname"), col("firstname")).alias("Member Name"), col("name").alias("Facility Name")).distinct()
    .orderBy("Member Name", "Facility Name")
)
# Load
result.write.mode("overwrite").partitionBy("Facility Name").parquet("dbfs:/FileStore/tables/threeJoin.parquet")

## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
# Extract
import requests
from pyspark.sql import Row
from pyspark.sql.functions import col
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
symbols = ["GOOGL", "AAPL", "MSFT", "TSLA"]

all_data = []

for symbol in symbols:
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "datatype": "json",
        "outputsize": "compact"
    }

    headers = {
        "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
        "X-RapidAPI-Key": "92d92f3032msh05701a5e9918f16p1555edjsnca05a8131ba6"
    }

    response = requests.get("https://alpha-vantage.p.rapidapi.com/query", headers=headers, params=querystring)
    data = response.json()
    time_series = data.get("Time Series (Daily)", {})
    
    for date, daily_data in time_series.items():
        all_data.append(Row(
            symbol=symbol,
            date=date,
            open=float(daily_data["1. open"]),
            high=float(daily_data["2. high"]),
            low=float(daily_data["3. low"]),
            close=float(daily_data["4. close"]),
            volume=int(daily_data["5. volume"])
        ))
stocks_df = spark.createDataFrame(all_data)
display(stocks_df)



symbol,date,open,high,low,close,volume
GOOGL,2026-01-28,336.06,337.535,331.94,336.01,27434434
GOOGL,2026-01-27,335.365,337.91,333.48,334.55,19722250
GOOGL,2026-01-26,327.805,335.84,327.0,333.26,26042120
GOOGL,2026-01-23,332.485,333.6889,327.45,327.93,27279974
GOOGL,2026-01-22,334.45,335.15,328.75,330.54,26253640
GOOGL,2026-01-21,320.92,332.4799,319.3518,328.38,35386603
GOOGL,2026-01-20,320.87,327.73,320.43,322.0,34808916
GOOGL,2026-01-16,334.41,334.65,327.7,330.0,40341637
GOOGL,2026-01-15,337.65,337.69,330.74,332.78,28442406
GOOGL,2026-01-14,335.06,336.52,330.48,335.84,28525571


In [0]:
from pyspark.sql.functions import col, max as spark_max, weekofyear, to_date


# Transform
stocks_df = stocks_df.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
stocks_df = stocks_df.withColumn("week", weekofyear(col("date")))

max_weekly_df = stocks_df.groupBy("symbol", "week").agg(spark_max(col("close")).alias("max_close")).orderBy("symbol","week")
display(max_weekly_df)


symbol,week,max_close
AAPL,1,273.76
AAPL,2,267.26
AAPL,3,261.05
AAPL,4,248.35
AAPL,5,258.27
AAPL,36,239.69
AAPL,37,237.88
AAPL,38,245.5
AAPL,39,256.87
AAPL,40,258.02


In [0]:
# Load
max_weekly_df.write.format("delta").mode("overwrite").partitionBy("symbol").saveAsTable("max_closing_price_weekly")

## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
# Extract

rna_df = (spark.read
  .format("jdbc")
  .option("url", "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs")
  .option("dbtable", "(SELECT * FROM rna LIMIT 100) AS rna_subquery")
  .option("user", "reader")
  .option("password", "NWDMCE5xdipIjRrp")
  .load()
)

display(rna_df)

id,upi,timestamp,userstamp,crc64,len,seq_short,seq_long,md5
8988357,URS00008926C5,2015-10-20T18:04:07.000Z,RNACEN,F9626977AB4E17FB,1336,TCAGCGGCGAACGGGTGAGTAACACGTGGGTGACTTGCCCCGAAGATGGGGATAACCTCTGGAAACGGGGGCTAATACCCAATGTGCTCGGTGATTCGGTTCATCGAGTAAAGCTCCGGCGCTTCGGGAGAGGCCTGCGGCCCATCAGCTAGTTGGTAGGGTAACGGCCTACCAAGGCAGAGGCGGGTAGGGGGCGTGAGAGCGCGGACCCCCACACTGGCACTGAGATACGGGCCAGACTCCTACGGGAGGCAGCAGTAAGGGATATTGCGCAATGGACGAAAGTCAGACGCAGCGACGCCGCGTGGGCGATGAAGGCCTTCGGGTTGTAAAGCCCTTTTATGGGGGAAGAGAAAAAGGACGGTACCCCAGGAATAAGTCCCGGCTAACTACGTGCCAGCAGCCGCGGTAAAACGTAGGGGACAAGCGTTATCCGGATTCACTGGGCGTAAAGAGCGTTGAGGCGGTTCCGTAAGTTGGGCGTGAAAGCTCCGGGCTTAACTCGGAGATGTCGTTCAATACTGCGGGGCTTGAGGACAGCAGAGGAAGGTGGAATTCCCGGTGTAGTGGTGAAATGCGTAGATATCGGGAGGAACACCCGTGGCGAAGGCGGCCTTCTGGGCTGTTCCTGACGCTGAAGGCGAAAGCTAGGGGAGCGAACGGGATTAGATACCCCGGTAGTCCTAGCTGTAAACGATGGATGCTGGGTGTGGGGGGTGTAAATTCCCTCTGTGCCGAAGCAAACGCGTTAAGCATCCCGCCTGGGGACTACGGCCGCAAGGCTAAAACTCAAACGAATTGACGGGGGCCCGCACAAGCAGCGGAGCGTGTGGTTTAATTCGATGCTACACGAAGAACCTTACCTGGGTTTGACATGCACGTGGTAGGGAACCGAAAGGGGACCGACCTTCGGGAGCGTGCACAGGTGCTGCATGGCTGTCGTCAGCTCGTGCCGTGAGGTGTCGGGTTAAGTCCCGTAACGAGCGCAACCCTTGCCCTTAGTTACAAGTGTCTAAGGGGACTGCCCGGGACAACTGGGAGGAAGGTGGGGATGACGTCAAGTCAGCATGGCCTTTATATCCAGGGCTACACACACGCTACAATGGCCGGTACAATAGGTTGCGAAGTCGTGAGGCGGAGCCAATCCTCAAAGCCGGTCTCAGTTCGAATTGCAGTCTGCAACTCGACTGCATGAAGCTGGAGTTGCTAGTAATCGCAGGTCAGCTATACTGCGGTGATACGTTCCCGGGCCTTGTACACACCGCCCGTCACGTCATGGAAGCTGGCAACGCCTGAAGCCGGTGAGCTAACCCGAAAGGGAGGCAGCCGTCGAGGG,null,fe4792a9218a34fdee33c9c52c548cf7
8988360,URS00008926C8,2015-10-20T18:04:07.000Z,RNACEN,DEA611A8ABDE9078,1307,ACTGCTATCGGATTGATACTAAGCCATGCGAGTCATTGTAGCAATACAAGGCATACGGCTCAGTAACGCGTAGTCAACCTAACCTATGGACGGGAATAACCTCGGGAAACTGAGAATAATGCCCGATAGAACATTATGCCTGGAATGGTTTATGTTCCAAATGATTTATCGCCGTAGGATGGGACTGCGGCCTATCAGTTTGTTGGTGAGGTAATGGCCCACCAAGACTATTACAGGTACGGGCTCTGAGAGGAGTAGCCCGGAGATGGGTACTGAGACACGGACCCAGGCCCTATGGGGCGCAGCAGGCGAGAAAACTTTGCAATGTGCGAAAGCACGACAAGGTTAATCCGAGTGATTTGTGCTAAACGAATCTTTTGTTAGTCCTAGAAACACTAACGAATAAGGGGTGGGCAAGTTCTGGTGTCAGCCGCCGCGGTAAAACCAGCACCTCAAGTGGTCAGGATGATTATTGGGCCTAAAGCATCCGTAGCCGGCCCTGTAAGTTTTCGGTTAAATCTGTACGCTTAACGTACAGGCTGCCGGGAATACTGCAGAGCTAGGGAGTGGGAGAAGTAGACGGTACTCGGTAGGAAGTGGTAAAATGCTTTGATCTATCGATGACCACCTGTGGCGAAGGCGGTCTACTAGAACACGTCCGACGGTGAGGGATGAAAGCTGGGGGAGCAAACCGGATTAGATACCCGGGTAGTCCCAGCTGTAAACTATGCAAACTCAGTGATGCATTGGCTTGTGGCCAATGCAGTGCTGCAGGGAAGCCGTTAAGTTTGCCGCCTGGGAAGTACGTACGCAAGTATGAAACTTAAAGGAATTGGCGGGGGAGCACCACAAGGGGTGAAGCCTGCGGTTCAATTGGAGTCAACGCCAGAAATCTTACCCGGAGAGACAGCAGAATGAAGGTCAAGCTGAAGACTTTACCAGACAAGCTGAGAGGTGGTGCATGGCCGTCGCCAGCTCGTGCCGTGAGATGTCCTGCTAAGTCAGGTAACGAGCGAGATCCCTGCCTCTAGTTGCCACCATTACTCTCAGGAGTAGTGGGGCGAATTAGCGGGACCGCCGCAGTTAATGCGGAGGAAGGAAGGGGCCACGGCAGGTCAGTATGCCCCGAAACTCTGGGGCCACACGCGGGCTGCAATGGTAACGACAATTGGTTTCGAATCCGAAAGGATGAGGTAATCCTCAAACGTTACCACAGTTATGACTGAGGGCTGCAACTCGCCCTCACGAATATGGAATCCCTAGTAACTGCGTGTCATTATCGCGCGGTGAATACGTCCCTGCTCCTT,null,5eb946fc85a2e16f40b2de67dbff627b
8988361,URS00008926C9,2015-10-20T18:04:07.000Z,RNACEN,AE161A21AF6713C0,1367,AGCCCAGCTTGCTGGGTGGATTAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTCTGGGATAAGCCTGGGAAACTGGGTCTAATACCGGATAGGAACGTCCACCGCATGGTGGGTGTTGGAAAGATTTATCGGTCATGGATGGACTCGCGGCCTATCAGCTTGTTGGTGAGGTAATGGCTCACCAAGGCGACGACGGGTAGCCGGCCTGAGAGGGTGACCGGCCACACTGGGACTGAGACACGGCCCAGACTCCTACGGGAGGCAGCAGTGGGGAATATTGCACAATGGGCGAAAGCCTGATGCAGCGACGCCGCGTGAGGGATGACGGCCTTCGGGTTGTAAACCTCTTTCAGTAGGGAAGAAGCGAAAGTGACGGTACCTGCAGAAGAAGCACCGGCTAACTACGTGCCAGCAGCCGCGGTAATACGTAGGGTGCGAGCGTTATCCGGAATTATTGGGCGTAAAGAGCTCGTAGGCGGTTTGTCGCGTCTGTCGTGAAAGTCCGGGGCTTAACCCCGGATCTGCGGTGGGTACGGGCAGACTAGAGTGCAGTAGGGGAGACTGGAATTCCTGGTGTAGCGGTGGAATGCGCAGATATCAGGAGGAACACCGATGGCGAAGGCAGGTCTCTGGGCTGTAACTGACGCTGAGGAGCGAAAGCATGGGGAGCGAACAGGATTAGATACCCTGGTAGTCCATGCCGTAAACGTTGGGCACTAGGTGTGGGGACCATTCCACGGTTTCCGCGCCGCAGCTAACGCATTAAGTGCCCCGCCTGGGGAGTACGGCCGCAAGGCTAAAACTCAAAGGAATTGACGGGGGCCCGCACAAGCGGCGGAGCATGCGGATTAATTCGATGCAACGCGAAGAACCTTACCAAGGCTTGACATGTTCTCGATCGCCGTAGAGATACGGTTTCCCCTTTGGGGCGGGATCACAGGTGGTGCATGGTTGTCGTCAGCTCGTGTCGTGAGATGT

In [0]:
# Load
rna_df.write.format("delta").mode("overwrite").saveAsTable("rna")